# Phase 12.4 — Search Evaluation Runner

Load the benchmark, run a deterministic mapping executor, and inspect metrics + lineage.
No ProductIQ retrieval or database.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.retrieval.evaluation.search_evaluation import (
    MappingSearchEvaluationExecutor,
    SearchEvaluationVariant,
    SearchMetricConfiguration,
    build_search_evaluation_request,
    default_search_benchmark_path,
    load_search_evaluation_benchmark,
    run_search_evaluation,
)

In [ ]:
benchmark = load_search_evaluation_benchmark(default_search_benchmark_path(ROOT))
variant = SearchEvaluationVariant(
    variant_name="notebook_toy_executor",
    variant_version="0.0.0",
    description="Returns judged IDs in label order (not real search).",
)
config = SearchMetricConfiguration(k_values=(1, 5, 10), evaluation_top_k=10, execution_top_k=20)
request = build_search_evaluation_request(benchmark, variant, metric_configuration=config)

rankings = {
    query.query_id: tuple(j.product_id for j in query.relevance_judgments)
    for query in benchmark.queries
}
executor = MappingSearchEvaluationExecutor(rankings)
result = run_search_evaluation(request, executor)
print("variant:", result.lineage.variant_name)
print("queries:", result.aggregate.query_count)
print("mrr:", result.aggregate.mrr)
print("mean precision@5:", result.aggregate.mean_precision_at_k[5])

In [ ]:
first_row = result.per_query[0]
print("first query:", first_row.query_id)
print("precision@1:", first_row.precision_at_k[1])
print("lineage catalog:", result.lineage.catalog_artifact)